<a href="https://colab.research.google.com/github/Bima34157/data-science-2026/blob/main/pertemuan_3_zaldi_bima_aditya_220401010271.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Pertemuan ke-3

- Nama    : Zaldi Bima Aditya
- NIM     : 220401010271
- Kelas   : IF405

In [2]:
# ============================================
# PIPELINE DATA CLEANING - HOUSING DATASET
# ============================================

import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import missingno as msno
import requests
from pandas import json_normalize
import matplotlib.pyplot as plt

In [8]:
# Load Dataset & Eksplorasi Awal

url = "https://drive.google.com/uc?export=download&id=17PgEvXH6DaPVz5PcdQ7Rvt1LauOpCRXS"
df = pd.read_csv(url)

print("Dataset berhasil dimuat!")
print("Shape awal:", df.shape)
print(df.isnull().sum())


Dataset berhasil dimuat!
Shape awal: (130, 7)
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


In [10]:
#1 Hapus Duplikat
df.drop_duplicates(inplace=True)
print("Shape setelah hapus duplikat:", df.shape)

Shape setelah hapus duplikat: (130, 7)


In [15]:
#2 Normalisasi String

# Kolom kota
df['kota'] = df['kota'].str.strip().str.title()

# Kolom kondisi:
df['kondisi'] = df['kondisi'].str.strip().str.lower()

# Cek hasil
print("Kota unik setelah normalisasi:")
print(sorted(df['kota'].unique()))

print("\nKondisi unik setelah normalisasi:")
print(sorted(df['kondisi'].unique()))

Kota unik setelah normalisasi:
['Bandung', 'Bdg', 'Depok', 'Dpk', 'Jakarta', 'Jogja', 'Makassar', 'Mdn', 'Medan', 'Mksr', 'Sby', 'Semarang', 'Smg', 'Surabaya', 'Ygy', 'Yogyakarta']

Kondisi unik setelah normalisasi:
['bagus', 'baik', 'baik sekali', 'cukup', 'jelek', 'perlu renovasi', 'rusak', 'sedang']


In [17]:
#3 Imputasi Missing Values

print("Missing values sebelum imputasi:")
print(df.isnull().sum())


df['luas_m2']    = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())


df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])


if df['kota'].isnull().sum() > 0:
    df['kota'] = df['kota'].fillna(df['kota'].mode()[0])
if df['kondisi'].isnull().sum() > 0:
    df['kondisi'] = df['kondisi'].fillna(df['kondisi'].mode()[0])

print("\nMissing values setelah imputasi:")
print(df.isnull().sum())
print("\nImputasi missing values selesai.")

Missing values sebelum imputasi:
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64

Missing values setelah imputasi:
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64

Imputasi missing values selesai.


In [19]:
#4 Tangani Outlier dengan IQR

kolom_outlier = ['harga_juta', 'luas_m2']

for kolom in kolom_outlier:
    Q1 = df[kolom].quantile(0.25)
    Q3 = df[kolom].quantile(0.75)
    IQR = Q3 - Q1

    lower_fence = Q1 - 1.5 * IQR
    upper_fence = Q3 + 1.5 * IQR

    print(f"\n{kolom}")
    print(f"Lower Fence : {lower_fence:.2f}")
    print(f"Upper Fence : {upper_fence:.2f}")

    jumlah_outlier = ((df[kolom] < lower_fence) | (df[kolom] > upper_fence)).sum()
    print(f"Jumlah Outlier : {jumlah_outlier}")

    # Menghapus outlier
    df = df[(df[kolom] >= lower_fence) & (df[kolom] <= upper_fence)]

print("\nJumlah data setelah outlier dihapus:", len(df))


harga_juta
Lower Fence : -422.75
Upper Fence : 1719.25
Jumlah Outlier : 0

luas_m2
Lower Fence : -145.22
Upper Fence : 512.97
Jumlah Outlier : 0

Jumlah data setelah outlier dihapus: 130


In [24]:
#5 Validasi & Ekspor

df.to_csv('housing_clean.csv', index=False)
print("Dataset bersih tersimpan sebagai 'housing_clean.csv'")
print(f"Shape final: {df.shape}")

Dataset bersih tersimpan sebagai 'housing_clean.csv'
Shape final: (130, 7)


In [26]:
#6 Akses API JSONPlaceholder

import requests
import pandas as pd

# Mengakses API
url = 'https://jsonplaceholder.typicode.com/posts'
response = requests.get(url)

# Mengubah respons JSON menjadi DataFrame
df_api = pd.DataFrame(response.json())

# Menampilkan 5 data pertama
print(df_api.head())

# Informasi DataFrame
print("\nJumlah data:", len(df_api))
print("\nKolom:")
print(df_api.columns.tolist())

   userId  id                                              title  \
0       1   1  sunt aut facere repellat provident occaecati e...   
1       1   2                                       qui est esse   
2       1   3  ea molestias quasi exercitationem repellat qui...   
3       1   4                               eum et est occaecati   
4       1   5                                 nesciunt quas odio   

                                                body  
0  quia et suscipit\nsuscipit recusandae consequu...  
1  est rerum tempore vitae\nsequi sint nihil repr...  
2  et iusto sed quo iure\nvoluptatem occaecati om...  
3  ullam et saepe reiciendis voluptatem adipisci\...  
4  repudiandae veniam quaerat sunt sed\nalias aut...  

Jumlah data: 100

Kolom:
['userId', 'id', 'title', 'body']
